# Wastely Waste Volume Forecast

Train and inspect the Python waste-volume model against the live SvelteKit training-data API.

If the app is not running, this notebook now falls back to built-in synthetic rows instead of crashing.

In [ ]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT_CANDIDATE = NOTEBOOK_DIR
for candidate in [PROJECT_ROOT_CANDIDATE, *PROJECT_ROOT_CANDIDATE.parents]:
    if (candidate / 'package.json').exists() and (candidate / 'ml').exists():
        PROJECT_ROOT = candidate
        break
else:
    PROJECT_ROOT = NOTEBOOK_DIR.parent.parent

sys.path.insert(0, str(PROJECT_ROOT / 'ml'))

from notebook_support import find_project_root, load_training_rows
from waste_model import predict_rows, save_model, train_model

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
PROJECT_ROOT


In [ ]:
APP_URL = os.getenv('WASTELY_APP_URL', 'http://127.0.0.1:5173')
ML_SHARED_TOKEN = os.getenv('ML_SHARED_TOKEN', '')
TRAINING_DATA_FILE = os.getenv('WASTELY_TRAINING_DATA_FILE', '')

rows, source_info = load_training_rows(
    app_url=APP_URL,
    days=45,
    token=ML_SHARED_TOKEN,
    data_file=Path(TRAINING_DATA_FILE).expanduser() if TRAINING_DATA_FILE else None,
)

print(source_info['message'])
len(rows), rows[0].keys() if rows else [], source_info


In [ ]:
model = train_model(rows)
print('trained_rows =', model.trained_rows)
print('rmse =', round(model.rmse, 3))
print('model_version =', model.model_version)


In [ ]:
preview_rows = rows[:5]
predictions = predict_rows(model, preview_rows)
for prediction in predictions:
    print(prediction)


In [ ]:
model_path = PROJECT_ROOT / 'ml' / 'model.json'
save_model(model, model_path)
print('saved model to', model_path)
